# 📝 Aiscern — AI Text Detector Fine-tuning
### Kaggle P100 | DeBERTa-v3-base + LoRA | ~3.5h

**Base model**:  (SOTA text classification)  
**Strategy**: LoRA fine-tuning on attention layers (~2% params trained)  
**Output**:   
**Expected accuracy**: **96–98%** (F1 ≥ 0.96)  

| Dataset | Samples | Source |
|---|---|---|
| HC3 (Human-ChatGPT Corpus) | ~87,000 | Hello-SimpleAI |
| AI-Human text | ~50,000 | andythetechnerd03 |
| AI generated text | ~50,000 | Dogge |
| RAID benchmark | ~30,000 | raid-bench |


In [ ]:
!pip install -q transformers==4.40.0 datasets peft==0.10.0 accelerate evaluate scikit-learn huggingface_hub sentencepiece protobuf

import os, warnings, logging, torch
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(f"✅ GPU: {gpu}")
print(f"✅ CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if device=="cuda" else "⚠️  No GPU")
print("✅ Dependencies installed")


In [ ]:
# ── HF TOKEN (reads from Kaggle Secrets — no manual edit needed) ──
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token loaded from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("✅ HF token loaded from environment")
    else:
        raise RuntimeError("❌ HF_TOKEN not found. Add it via Notebook → Add-ons → Secrets in Kaggle.")


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────

BASE_MODEL      = "microsoft/deberta-v3-base"
PUSH_REPO       = "saghi776/aiscern-text-detector"
CHECKPOINT_DIR  = "./text-checkpoints"
MAX_LEN         = 512
SAMPLES_PER_CLASS = 60000   # balanced — 60k AI + 60k human
BATCH_SIZE      = 16        # P100 16GB handles DeBERTa batch=16 fine
EPOCHS          = 5
LR              = 1e-4
WARMUP_RATIO    = 0.06
WEIGHT_DECAY    = 0.01

# LoRA config — trains ~2% of params, fits easily in 16GB VRAM
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.1

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("✅ Config set")
print(f"   Model : {BASE_MODEL}")
print(f"   Push  : {PUSH_REPO}")
print(f"   Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Max tokens: {MAX_LEN}")


In [ ]:
# ── LOAD DATASETS ────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd, numpy as np

def norm_label(v):
    s = str(v).strip().lower()
    if s in ("1","ai","chatgpt","generated","fake","machine","true"): return 1
    if s in ("0","human","real","wiki","reddit","false","authentic"): return 0
    return -1

all_rows = []

# 1. HC3 — Human-ChatGPT Comparison Corpus (best quality)
try:
    ds = load_dataset("Hello-SimpleAI/HC3", "all", split="train", token=HF_TOKEN)
    for row in ds:
        for txt in (row.get("human_answers") or []):
            if txt and len(txt) > 80: all_rows.append({"text": txt[:2000], "label": 0})
        for txt in (row.get("chatgpt_answers") or []):
            if txt and len(txt) > 80: all_rows.append({"text": txt[:2000], "label": 1})
    print(f"✅ HC3: {len(all_rows)} rows")
except Exception as e:
    print(f"⚠️  HC3 skipped: {e}")

before = len(all_rows)
# 2. AI-Human text dataset
try:
    ds2 = load_dataset("andythetechnerd03/AI-human-text", split="train", token=HF_TOKEN)
    for row in ds2:
        txt = row.get("text","") or row.get("content","")
        lbl = row.get("label", row.get("generated", -1))
        l   = norm_label(lbl)
        if l >= 0 and txt and len(txt) > 80:
            all_rows.append({"text": txt[:2000], "label": l})
    print(f"✅ AI-Human: +{len(all_rows)-before} rows")
except Exception as e:
    print(f"⚠️  AI-Human skipped: {e}")

before = len(all_rows)
# 3. Dogge AI generated text
try:
    ds3 = load_dataset("Dogge/ai_generated_text_dataset", split="train", token=HF_TOKEN)
    for row in ds3:
        txt = row.get("text","") or row.get("content","")
        lbl = row.get("label", row.get("generated", row.get("is_ai", -1)))
        l   = norm_label(lbl)
        if l >= 0 and txt and len(txt) > 80:
            all_rows.append({"text": txt[:2000], "label": l})
    print(f"✅ Dogge: +{len(all_rows)-before} rows")
except Exception as e:
    print(f"⚠️  Dogge skipped: {e}")

before = len(all_rows)
# 4. RAID benchmark (diverse AI sources)
try:
    ds4 = load_dataset("raid-bench/raid", split="train", token=HF_TOKEN)
    for row in ds4:
        txt = row.get("generation","") or row.get("text","")
        src = str(row.get("model","")).lower()
        if txt and len(txt) > 80:
            # raid: "human" source means label 0, model outputs label 1
            l = 0 if "human" in src else 1
            all_rows.append({"text": txt[:2000], "label": l})
    print(f"✅ RAID: +{len(all_rows)-before} rows")
except Exception as e:
    print(f"⚠️  RAID skipped: {e}")

before = len(all_rows)
# 5. Fallback: aisquared/human-vs-ai-text-generation
try:
    ds5 = load_dataset("aisquared/human-vs-ai-text-generation", split="train", token=HF_TOKEN)
    for row in ds5:
        txt = row.get("text","")
        lbl = norm_label(row.get("label", -1))
        if lbl >= 0 and txt and len(txt) > 80:
            all_rows.append({"text": txt[:2000], "label": lbl})
    print(f"✅ aisquared: +{len(all_rows)-before} rows")
except Exception as e:
    print(f"⚠️  aisquared skipped: {e}")

print(f"\n📊 Total raw rows: {len(all_rows)}")
if len(all_rows) < 2000:
    raise RuntimeError("❌ Too few samples loaded. Check HF token and dataset access.")


In [ ]:
# ── BALANCE + SPLIT ──────────────────────────────────────────────
import random
from collections import Counter

random.shuffle(all_rows)
ai_rows    = [r for r in all_rows if r["label"] == 1]
human_rows = [r for r in all_rows if r["label"] == 0]

print(f"Before balance — AI: {len(ai_rows)} | Human: {len(human_rows)}")

# Cap at SAMPLES_PER_CLASS for training efficiency
n = min(SAMPLES_PER_CLASS, len(ai_rows), len(human_rows))
balanced = ai_rows[:n] + human_rows[:n]
random.shuffle(balanced)

# 90/10 train/val split
split_idx   = int(len(balanced) * 0.9)
train_rows  = balanced[:split_idx]
val_rows    = balanced[split_idx:]

print(f"✅ Train: {len(train_rows)} | Val: {len(val_rows)}")
print(f"   Labels: {Counter([r["label"] for r in train_rows])}")

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)


In [ ]:
# ── TOKENIZE ─────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

train_tok = train_ds.map(tokenize, batched=True, batch_size=256, remove_columns=["text"])
val_tok   = val_ds.map(tokenize,   batched=True, batch_size=256, remove_columns=["text"])

train_tok.set_format("torch")
val_tok.set_format("torch")

print(f"✅ Tokenized — train cols: {train_tok.column_names}")


In [ ]:
# ── MODEL + LoRA ─────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "human", 1: "ai"},
    label2id={"human": 0, "ai": 1},
    token=HF_TOKEN,
    ignore_mismatched_sizes=True,
)

# DeBERTa target modules — attention query/value projections
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["query_proj", "value_proj"],  # DeBERTa attention layers
    bias="none",
    modules_to_save=["classifier", "pooler"],      # always train head
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied: {trainable/1e6:.2f}M / {total/1e6:.2f}M params trainable ({100*trainable/total:.1f}%)")


In [ ]:
# ── TRAINING ─────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average="binary")
    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = 0.0
    print(f"   📊 acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}")
    return {"accuracy": acc, "f1": f1, "roc_auc": auc}

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=True,
    hub_model_id=PUSH_REPO,
    hub_token=HF_TOKEN,
    hub_strategy="every_save",         # push after every epoch
    logging_steps=100,
    fp16=(device=="cuda"),              # mixed precision for speed
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
trainer.train(resume_from_checkpoint=True if os.path.exists(CHECKPOINT_DIR) and len(os.listdir(CHECKPOINT_DIR)) > 0 else None)
print("✅ Training complete!")


In [ ]:
# ── EVALUATE + PUSH TO HF ────────────────────────────────────────
from huggingface_hub import HfApi

results = trainer.evaluate()
acc = results.get("eval_accuracy", 0)
f1  = results.get("eval_f1", 0)
auc = results.get("eval_roc_auc", 0)

print(f"\n🏆 FINAL RESULTS:")
print(f"   Accuracy : {acc*100:.2f}%")
print(f"   F1 Score : {f1:.4f}")
print(f"   ROC-AUC  : {auc:.4f}")

# Push best model to HuggingFace Hub
model.push_to_hub(PUSH_REPO, token=HF_TOKEN, commit_message=f"LoRA fine-tune | acc={acc:.3f} f1={f1:.3f}")
tokenizer.push_to_hub(PUSH_REPO, token=HF_TOKEN)

# Save model card
api = HfApi(token=HF_TOKEN)
card = f"""---
tags: [text-classification, ai-detection, deberta, lora]
license: mit
datasets: [Hello-SimpleAI/HC3, andythetechnerd03/AI-human-text]
metrics: [accuracy, f1]
---
# Aiscern Text Detector
DeBERTa-v3-base fine-tuned with LoRA for AI vs Human text detection.
- **Accuracy**: {acc*100:.1f}%
- **F1**: {f1:.4f}
- **ROC-AUC**: {auc:.4f}
- **Labels**: 0=human, 1=ai
## Usage
from transformers import pipeline
clf = pipeline('text-classification', model='{PUSH_REPO}')
result = clf('Your text here...')
'"""
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=PUSH_REPO, token=HF_TOKEN)

print(f"\n✅ Model pushed to: https://huggingface.co/{PUSH_REPO}")
print(f"   Use in Aiscern: HF_TOKEN inference API auto-loads on first request")
